# Dunnhumby: N/V + 세부 유형 내 가격 M2 (개발 seed 43)

기존 M1과 기존 4축 M2 결과를 검증해 재사용하고 새 M2 **한 arm**만 학습합니다. N과 V를 모두 유지하며, 세부 상품 유형은 유형 내 학습구간 가격 백분위 계산에만 사용합니다. 이진 그래프·uniform K=1·plain BPR·한 optimizer, 최대 300 epoch/25 epoch 간격 평가입니다. 최종 test/holdout 결과로 해석하지 마세요.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_nv_modulated_id_seed43_v3/reports/result.json'
PREVIOUS = ROOT/'results_v3_dunnhumby_shared_feature_residual_m2_seed43_v1/reports/result.json'
# 기존 결과가 ZIP에 있다면 REPORT/PREVIOUS를 각각 ZIP 경로로 바꿀 수 있습니다.
try:
    if not ROOT.is_dir():
        drive.mount('/content/drive')
    if not ROOT.is_dir() or not REPORT.is_file() or not PREVIOUS.is_file():
        raise RuntimeError(f'기준 결과 확인: {REPORT} / {PREVIOUS}')
except (OSError, ValueError, NotImplementedError) as exc:
    raise RuntimeError('Drive 연결 실패. 이 셀부터 다시 실행하세요. 학습은 시작하지 않았습니다.') from exc
SOURCE_COMMIT = '668a0dfb1b3889cef529c5e9118f29c643fa9f04'
REPO = Path('/content/clv-m2-fine-price-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 코드가 로드됨: 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_m2_fine_price_screen as screen
OUT = ROOT/'results_v3_dunnhumby_m2_fine_price_seed43_v1'
assert screen.VERSION == 'm2-fine-type-price-nv-seed43-development-v1'
screen.read_source(REPORT, screen.configure(str(OUT)))
screen.read_source(REPORT, screen.configure(str(OUT)))
screen.read_previous(PREVIOUS, screen.configure(str(OUT)))
print('고정 코드 및 기존 M1·M2 결과 확인:', SOURCE_COMMIT)


In [ ]:
cfg, prepared = screen.prepare(REPORT, PREVIOUS, OUT)
assert cfg.seeds == (43,) and cfg.epochs == 300 and cfg.pref_reg == .001
assert [a['model_id'] for a in prepared['anchors']] == ['m1']
assert prepared['previous_arm']['model_id'] == screen.prior.MODEL_ID
assert screen.spec()['role'] == 'M2' and not screen.spec()['weighted']
print('새 학습:', screen.spec())
print('분리된 규제 계수:', prepared['feature_settings'])
print('선택 규칙:', screen.selection(cfg))


In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))


In [ ]:
import pandas as pd
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
report_path = OUT/'reports/result.json'
paths = {}
if report_path.is_file():
    report = json.loads(report_path.read_text())
    assert report['code_version'] == screen.VERSION
    paths = report['paths']
    print(pd.read_csv(paths['absolute']).set_index('model_id').T.to_string())
    print(json.dumps(report['reading'], ensure_ascii=False, indent=2))
    print('M1 및 기존 M2와 전체 비교:', paths['comparison'])
    print('같은 epoch 비교:', paths['same_epoch_comparison'])
    print('특징 작동:', paths['diagnostics'])
else:
    print('완료 결과 없음: 저장된 준비 진단만 내려받습니다. 성과 판정은 보류합니다.')
zip_path = Path('/content/m2_fine_price_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
    diagnostic = OUT/'feature_diagnostic.json'
    if diagnostic.is_file():
        archive.write(diagnostic, arcname=diagnostic.name)
files.download(str(zip_path))
